# Paper Figures

In [ ]:
### import packages using the scviEnv environment
import custom_functions_scviEnv as cf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import warnings

sc.set_figure_params(scanpy=True, dpi=80, dpi_save=150, frameon=True, vector_friendly=True, 
                     fontsize=14, figsize=None, color_map=None, format='png', facecolor=None, 
                     transparent=False, ipython_format='png2x')

### Set font to Arial for all plots
plt.rcParams['font.family'] = 'Arial'

### Ignores warning mesaages
warnings.filterwarnings("ignore")

## Loading in the data

In [ ]:
adata = sc.read_h5ad('../data/adata_all_samples_annotated.h5ad')
adata.obs.celltype.value_counts()

## Figure 4A

In [ ]:
###Plotting single graph of all low-level cells
fig, ax = plt.subplots(figsize = (10,10))
sc.pl.umap(adata, ax = ax, color='celltype', add_outline=True, size = 20, 
        legend_loc='on data', legend_fontsize=14, legend_fontoutline=2,frameon=False,
        title='Low-Level Annotation UMAP', palette = cf.bright_palette, show = False)

### Saved as a pdf for futher editing of the legends in Adobe Illustrator.
plt.savefig('../output/Figure_4A_umap_all_low_level_celltypes.pdf')        
plt.show()

## Figure 4B

In [ ]:
### Low-level cell type proportions plot using the same colors as the UMAP.

list = ['Adipocytes_White', 'Adipocytes_Brown', 'Endothelial_Cells', 'Fibroblasts', 'Immune_Cells', 'Mesothelial_Cells', 'Schwann_Cells', 'Pericytes_SMCs','Unknown']
color = dict(zip(list, cf.bright_palette))

# Adjust font sizes
plt.rcParams.update({'font.size': 16,   # Increase font size for tick labels, axis labels, and legend
                     'axes.titlesize': 22,  # Increase font size for title
                     'axes.labelsize': 20,  # Increase font size for axis labels
                     'xtick.labelsize': 20, # Increase font size for x-axis tick labels
                     'ytick.labelsize': 16})# Increase font size for y-axis tick labels
# Get the colors used in the UMAP plot
umap_colors = adata.uns['celltype_colors']

fig, ax = plt.subplots(figsize = (16,6))

### Create dictionary of proportions
props_dict = {"sample_id":[], "Cell Type":[], "Proportion":[],"Color": []}
for k in np.unique(adata.obs.sample_id):
    N = len(adata[adata.obs.sample_id == k].obs)
    for i in np.unique(adata[adata.obs.sample_id == k].obs.celltype):
        props_dict["sample_id"].append(k)
        i_len = len(adata[(adata.obs.celltype == i) & (adata.obs.sample_id == k)])
        props_dict["Cell Type"].append(i)
        props_dict["Proportion"].append(i_len/N)
        props_dict["Color"].append(color.get(i))
props_df = pd.DataFrame(props_dict)

sum_props_df = pd.DataFrame(props_df.groupby(["Cell Type", "Color"]).mean(numeric_only=True).sort_values("Proportion", ascending=False))
cell_type_order = sum_props_df.index.get_level_values("Cell Type").tolist()
palette = sum_props_df.index.get_level_values("Color").tolist()

### Plotting functions
ax = sns.barplot(x = "Cell Type", y = "Proportion", data = props_df, palette = palette,
            order = cell_type_order, capsize = 0.06, edgecolor = '0.2', lw = 2.5, errorbar='se', err_kws={'color': '0.2', 'linewidth': 2}
            )

kwargs = {'edgecolor':'0.2', 'linewidth':2.5, 'fc': 'none'}

ax = sns.swarmplot(data = props_df, x = 'Cell Type', y = 'Proportion',
                   dodge = 0.4, marker = 's', s = 3, order = cell_type_order, **kwargs)

### Aesthetics
plt.xticks(size = 16, ha = 'center', color = '0.2', rotation = 25) #weight = 'bold', 
plt.yticks(size = 16,  color = '0.2') #weight = 'bold',

ax.tick_params(width = 2.5, color = '0.2')

plt.xlabel('')
plt.ylabel('Proportion', size = 18, color = '0.2') #weight = 'bold'

# plt.title('Cell Type Proportions', fontweight = 'bold')  # Add title
ax.grid(False)

### After plotting
bars = ax.patches

for bar, proportion in zip(bars, props_df.groupby('Cell Type').mean(numeric_only=True).sort_values(by = "Proportion", ascending=False)["Proportion"]):
    # Get bar position
    x = bar.get_x() + bar.get_width() / 2
    #y = bar.get_height()
    
    # Label with the *original* proportion (not log)
    ax.text(x, 0, f"{proportion:.2%}",  # format as percent with 2 decimals
            ha='center', va='bottom', fontsize=18, rotation=0, color = 'white') #weight = 'bold',

plt.savefig('../output/Figure_4B_low_level_proportions_labeled.png', bbox_inches = "tight")
plt.show()

## Figure 4C

In [ ]:
df = cf.build_prop_df(adata, 'celltype')

labels = ['Thoracic Aorta', 'Mesentery', 'Kidney']
cf.proportion_plot_across_samples(df, labels, 'Figure_4C_low_level_proportions_barplot')

## Figure 4D

In [ ]:
fig, ax = plt.subplots(figsize = (14,6))

sc.pl.rank_genes_groups_matrixplot(
    adata,
    ax = ax,
    n_genes=4,
    values_to_plot="logfoldchanges",
    cmap='bwr',
    vmin=-10,
    vmax=10,
    min_logfoldchange=3,
    colorbar_title='log fold change',
    show = False
)

plt.savefig('../output/Figure_4D_low_level_marker_genes_matrix_plot.svg', bbox_inches = "tight")
plt.show()

## Figure 4E

In [ ]:
### Violin plot of gene expression for labeling "low-level" cell-types
mgl = ["Ucp1",   #Adipocytes_Brown 
       "Car3",  #Adipocytes_White
       "Cdh13",  #Endothelial_Cells
       "Fbn1",    #ASPCs
       "Mrc1",  #Immune_Cells
       "Gpm6a",    #Mesothelial_Cells
       "Myh11",   #SMCs
       "Scn7a",  #Schwann_Cells
       "Apoo"   #Unknown
]
ax = sc.pl.stacked_violin(adata, 
                          mgl, 
                          use_raw = True,
                          groupby='celltype', 
                          swap_axes = False, 
                          row_palette = cf.bright_palette,
                          show = False)

plt.savefig('../output/Figure_4E_low_level_marker_genes_violin_plot.svg', bbox_inches = "tight")
plt.show()


## Figure 6A

In [ ]:
mgl= ['Col1a1', 'Col1a2', 'Col3a1', 'Col4a1', 'Col4a2', 'Col5a1', 
      'Col5a2', 'Col5a3', 'Col6a1', 'Col6a2', 'Col6a3', 'Col6a6', 'Col12a1','Col14a1']

In [ ]:
sc.pl.dotplot(adata, mgl, groupby='celltype', swap_axes=False, use_raw = True, show = False, save = False)
plt.savefig('../output/Figure_6A_dotplot_collagen_genes_by_celltype.svg', bbox_inches = 'tight')

## Figure 6C

In [ ]:
mgl =  'Ano1,Asic1,Asic2,Cav1,ddr2,Fyn,itgb1,itgb4,piezo1,piezo2,Slc12a2,Tmem120a,Yap1,zyx'

mgl = mgl.lower()
mgl = mgl.replace(" ", "")
mgl = mgl.split(',')

for i in range(len(mgl)):
    mgl[i] = mgl[i].capitalize()

In [ ]:
sc.pl.dotplot(adata, mgl, groupby='celltype', swap_axes=False, use_raw = True, show = False, save = False)
plt.savefig('../output/Figure_6C_dotplot_mechanotransducer_genes_by_celltype.svg', bbox_inches = 'tight')